In [ ]:
import pandas as pd

# Load dataset change the path file
file_path = "path/amazon.csv"
df = pd.read_csv(file_path)

print(df.head())

                                               title   seller_name  \
0  LUNARM 2 PCS Tracing Wheel - Professional Stit...        ******   
1  MyLifeUNIT Paint Brush Washer, Airtight Stainl...  MYL***UNI***   
2  LET'S RESIN Epoxy Resin Dye,15 Color Transluce...        ******   
3  HTVRONT Tie Dye Kit - 32 Vibrant Colors Pre-Fi...  Dem***tor***   
4  Boon Lawn Countertop Baby Bottle Drying Rack B...  Ama***.co***   

         brand                                        description  \
0       LUNARM  LUNARM 2 PCS Tracing Wheel Package includes 2 ...   
1   MyLifeUNIT  Paint Brush Cleaner, Airtight Stainless Steel ...   
2  LET'S RESIN  Total 15 beautiful translucent colors of epoxy...   
3      HTVRONT  Explore Colorful Worlds With HTVRONT Tie Dye K...   
4         Boon  Dry it all! Boon brings three of their most po...   

   initial_price  final_price currency availability  reviews_count  \
0           6.49         6.49      USD     In Stock           1852   
1          14.99        

In [11]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 66 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   title                    1000 non-null   object 
 1   seller_name              1000 non-null   object 
 2   brand                    1000 non-null   object 
 3   description              1000 non-null   object 
 4   initial_price            999 non-null    float64
 5   final_price              999 non-null    float64
 6   currency                 1000 non-null   object 
 7   availability             1000 non-null   object 
 8   reviews_count            1000 non-null   int64  
 9   categories               790 non-null    object 
 10  asin                     1000 non-null   object 
 11  buybox_seller            1000 non-null   object 
 12  number_of_sellers        1000 non-null   int64  
 13  root_bs_rank             1000 non-null   int64  
 14  answered_questions       

In [40]:
# Check column names
print(df.columns)

Index(['asin', 'title', 'brand', 'rating', 'image_url', 'description',
       'user_id', 'user_rating'],
      dtype='object')


In [13]:
df['description'].fillna('', inplace=True)
df['categories'].fillna('Unknown', inplace=True)
df['rating'].fillna(df['rating'].mean(), inplace=True)  # Fill missing ratings with the mean

C:\Users\kakol\AppData\Local\Temp\ipykernel_936\11978349.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['description'].fillna('', inplace=True)
C:\Users\kakol\AppData\Local\Temp\ipykernel_936\11978349.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doi

In [14]:
df['tags'] = df['description'] + ' ' + df['features'] + ' ' + df['product_description']

In [26]:
import numpy as np

# Simulate user IDs and ratings
df['user_id'] = np.random.randint(1, 100, size=len(df))  # Random user IDs
df['user_rating'] = np.random.uniform(1, 5, size=len(df))  # Random ratings

In [15]:
# Check for missing values
print(df.isnull().sum())

title                     0
seller_name               0
brand                     0
description               0
initial_price             1
                       ... 
downloadable_videos       2
editorial_reviews      1000
about_the_author       1000
sponsered                 0
tags                     13
Length: 67, dtype: int64


In [16]:
# Check for duplicates
print(df.duplicated().sum())

0


In [21]:
# Drop duplicates
df.drop_duplicates(inplace=True)

# Content-Based Filtering

## Preprocess the Text Data

In [17]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
import re

# Download stopwords (if not already downloaded)
nltk.download('stopwords')

# Combine relevant text columns into a single column
df['tags'] = df['description'] + ' ' + df['features'] + ' ' + df['product_description']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kakol\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [18]:
df.head()

,title,seller_name,brand,description,initial_price,final_price,currency,availability,reviews_count,categories,...,customer_says,sustainability_features,climate_pledge_friendly,videos,other_sellers_prices,downloadable_videos,editorial_reviews,about_the_author,sponsered,tags
0,LUNARM 2 PCS Tracing Wheel - Professional Stit...,******,LUNARM,LUNARM 2 PCS Tracing Wheel Package includes 2 ...,6.49,6.49,USD,In Stock,1852,"[""Arts, Crafts & Sewing"",""Sewing"",""Sewing Noti...",...,"Customers find the product functional, well-ma...",NaN,False,"[""https://www.amazon.com/vdp/03f16646fe8e4c61a...",NaN,"[""https://m.media-amazon.com/images/S/vse-vms-...",NaN,NaN,True,LUNARM 2 PCS Tracing Wheel Package includes 2 ...
1,"MyLifeUNIT Paint Brush Washer, Airtight Stainl...",MYL***UNI***,MyLifeUNIT,"Paint Brush Cleaner, Airtight Stainless Steel ...",14.99,14.99,USD,In Stock,680,"[""Arts, Crafts & Sewing"",""Painting, Drawing & ...",...,Customers appreciate the caddy's leak-proof li...,NaN,False,"[""https://www.amazon.com/vdp/07b7bade607347678...",NaN,"[""https://m.media-amazon.com/images/S/vse-vms-...",NaN,NaN,False,"Paint Brush Cleaner, Airtight Stainless Steel ..."
2,"LET'S RESIN Epoxy Resin Dye,15 Color Transluce...",******,LET'S RESIN,Total 15 beautiful translucent colors of epoxy...,9.99,9.99,USD,In Stock,6006,"[""Arts, Crafts & Sewing"",""Crafting"",""Sculpture...",...,Customers find the dyes pigmented and easy to ...,NaN,False,"[""https://www.amazon.com/vdp/02b9ca5377fb4399a...",NaN,"[""https://m.media-amazon.com/images/S/vse-vms-...",NaN,NaN,True,Total 15 beautiful translucent colors of epoxy...
3,HTVRONT Tie Dye Kit - 32 Vibrant Colors Pre-Fi...,Dem***tor***,HTVRONT,Explore Colorful Worlds With HTVRONT Tie Dye K...,22.99,22.99,USD,In Stock,1731,"[""Arts, Crafts & Sewing"",""Fabric Decorating"",""...",...,Customers find the tie dye kit easy to use and...,NaN,False,"[""https://www.amazon.com/vdp/0503e80f640f4c0cb...","[{""delivery"":""FREE delivery Tuesday, February ...","[""https://m.media-amazon.com/images/S/vse-vms-...",NaN,NaN,True,Explore Colorful Worlds With HTVRONT Tie Dye K...
4,Boon Lawn Countertop Baby Bottle Drying Rack B...,Ama***.co***,Boon,Dry it all! Boon brings three of their most po...,35.99,23.99,USD,In Stock,2762,Unknown,...,Customers find the drying rack functional and ...,NaN,False,"[""https://www.amazon.com/vdp/059285e86f214385b...","[{""delivery"":""Delivery Tuesday, March 4. Or fa...","[""https://m.media-amazon.com/images/S/vse-vms-...",NaN,NaN,True,Dry it all! Boon brings three of their most po...


In [19]:
# Fill missing values with empty strings
df['description'] = df['description'].fillna('')
df['features'] = df['features'].fillna('')
df['product_description'] = df['product_description'].fillna('')

In [20]:
def clean_text(text):
    # Check if the input is a string
    if not isinstance(text, str):
        return ''  # Return empty string for non-string inputs
    
    # Convert to lowercase
    text = text.lower()
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

In [21]:
# Combine relevant text columns into a single column
df['tags'] = df['description'] + ' ' + df['features'] + ' ' + df['product_description']

# Clean the text data
df['tags'] = df['tags'].apply(clean_text)

In [22]:
tfidf = TfidfVectorizer(max_features=5000)  

In [23]:
tfidf_matrix = tfidf.fit_transform(df['tags'])

# Check the shape of the TF-IDF matrix
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (1000, 5000)


In [24]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Check the shape of the cosine similarity matrix
print("Cosine Similarity Matrix Shape:", cosine_sim.shape)  

Cosine Similarity Matrix Shape: (1000, 1000)


In [25]:
def get_recommendations(product_id, cosine_sim_matrix, df, top_n=5):
    # Get the index of the product
    product_index = df[df['asin'] == product_id].index[0]
    
    # Get the similarity scores for the product
    sim_scores = list(enumerate(cosine_sim_matrix[product_index]))
    
    # Sort the products based on similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the top N most similar products (excluding itself)
    sim_scores = sim_scores[1:top_n+1]
    
    # Get the product indices
    product_indices = [i[0] for i in sim_scores]
    
    # Return the top N recommended products
    return df[['asin', 'title', 'brand', 'rating']].iloc[product_indices]

# Example: Get recommendations for a product with ASIN 'B01JE9IDCY'
recommendations = get_recommendations('B01JE9IDCY', cosine_sim, df, top_n=5)
print(recommendations)


           asin                                              title  \
197  B00J8HCPAY    Plaid 50557E Flat Brush, (2-Piece), Gold Taklon   
742  B07X864TW9  Transon Artist Paint Brush Set of 12 for Water...   
479  B07X9SRV4H  UPINS 30 Pcs Flat Paint Brushes, Small Brush B...   
918  B075L8LCTG  Golden Maple Detail Paint Brushes Set 10pcs Mi...   
521  B06XTMNKVF  Transon 8pcs Round Watercolor Paint Brush Set ...   

            brand  rating  
197         Plaid     4.7  
742       TRANSON     4.7  
479         UPINS     4.5  
918  golden maple     4.7  
521       TRANSON     4.7  


# Collaborative Filtering

In [26]:
import numpy as np

# Simulate user IDs and ratings
num_users = 100  # Number of synthetic users
df['user_id'] = np.random.randint(1, num_users + 1, size=len(df))  # Random user IDs
df['user_rating'] = np.random.uniform(1, 5, size=len(df))  # Random ratings between 1 and 5

# Display the updated DataFrame
print(df[['asin', 'user_id', 'user_rating']].head())

         asin  user_id  user_rating
0  B08941QJH8       99     4.300513
1  B01JE9IDCY       81     4.491080
2  B07QQKJN2X        1     4.493863
3  B09YC659T8       56     2.462094
4  B07YTBP57L       40     2.767008


In [27]:
from scipy.sparse import csr_matrix

# Create a user-item interaction matrix
user_item_matrix = df.pivot_table(index='user_id', columns='asin', values='user_rating', fill_value=0)

# Convert the matrix to a sparse format for efficiency
user_item_matrix_sparse = csr_matrix(user_item_matrix.values)

In [28]:
# Install the surprise library if you don't have it
!pip install scikit-surprise

from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split

# Prepare the data for surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['user_id', 'asin', 'user_rating']], reader)

# Split the data into training and testing sets
trainset, testset = train_test_split(data, test_size=0.25)

# Train the SVD model
model = SVD()
model.fit(trainset)

# Test the model
predictions = model.test(testset)


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
# Function to get top N recommendations for a user
def get_collaborative_recommendations(user_id, model, df, top_n=5):
    # Get a list of all product IDs
    all_product_ids = df['asin'].unique()
    
    # Get a list of product IDs the user has already interacted with
    interacted_products = df[df['user_id'] == user_id]['asin'].unique()
    
    # Predict ratings for products the user hasn't interacted with
    recommendations = []
    for product_id in all_product_ids:
        if product_id not in interacted_products:
            predicted_rating = model.predict(user_id, product_id).est
            recommendations.append((product_id, predicted_rating))
    
    # Sort the recommendations by predicted rating
    recommendations.sort(key=lambda x: x[1], reverse=True)
    
    # Get the top N recommendations
    top_recommendations = recommendations[:top_n]
    
    # Return the top N recommended products
    return df[df['asin'].isin([x[0] for x in top_recommendations])][['asin', 'title', 'brand', 'rating']].drop_duplicates()

# Example: Get recommendations for user_id = 74
user_id = 74
collab_recommendations = get_collaborative_recommendations(user_id, model, df, top_n=5)
print(collab_recommendations)

           asin                                              title  \
418  B07MB5PTNG  Munchkin® Extend™ Faucet Extender, 2 Count (Pa...   
519  B0964BVHQH  Café Specialty Drip Coffee Maker | 10-Cup Insu...   
592  B093XTHCZL  Mr. Coffee Iced and Hot Coffee Maker, Single S...   
754  B0BNDN87NX  Muchcute Micro Fineliner Drawing Art Pens: 12 ...   
946  B0B8JTLBBX  Bath Toys for Toddlers 1-3: 6 Packs Light-Up F...   

          brand  rating  
418    Munchkin     4.6  
519        Café     3.6  
592  Mr. Coffee     4.5  
754    KOUSICOO     4.5  
946    VIBOYLAR     4.3  


# hybrid recommendation system

In [30]:
from sklearn.preprocessing import MinMaxScaler

# Normalize content-based similarity scores
content_scores = cosine_sim  # From content-based filtering
content_scaler = MinMaxScaler()
content_scores_normalized = content_scaler.fit_transform(content_scores)

# Normalize collaborative filtering predicted ratings
collab_scores = np.array([pred.est for pred in predictions])  # From collaborative filtering
collab_scaler = MinMaxScaler()
collab_scores_normalized = collab_scaler.fit_transform(collab_scores.reshape(-1, 1))

In [31]:
# Function to get hybrid recommendations for a user and product
def get_hybrid_recommendations(user_id, product_id, df, model, cosine_sim, content_scaler, collab_scaler, top_n=5):
    # Get content-based recommendations
    content_recs = get_recommendations(product_id, cosine_sim, df, top_n=top_n)
    
    # Get collaborative filtering recommendations
    collab_recs = get_collaborative_recommendations(user_id, model, df, top_n=top_n)
    
    # Combine the recommendations
    hybrid_recs = pd.concat([content_recs, collab_recs]).drop_duplicates().head(top_n)
    
    return hybrid_recs

# Example: Get hybrid recommendations for user_id = 1 and product_id = 'B000123456'
user_id = 64
product_id = 'B0027A5E34'
hybrid_recommendations = get_hybrid_recommendations(user_id, product_id, df, model, cosine_sim, content_scaler, collab_scaler, top_n=5)
print(hybrid_recommendations)

           asin                                              title  \
298  B0BWHDBM1K  6 Packs Retractable Tape Measure + 2PCS Soft B...   
845  B07Z4LP271  Frisco Craft Transfer Tape for Heat Vinyl - Ir...   
267  B0BGGV77DY  4 Inch x 18 Feet Black Hook and Loop Tape with...   
876  B09SFW612M  ECOHomes Heavy Duty Strips for Couch Cushions ...   
147  B0BX21QFKJ  Baby Proofing, Clear Edge Protector Strip, Sof...   

            brand  rating  
298  SmaringRobot     4.7  
845  Frisco Craft     4.5  
267        Nurkul     4.5  
876      ECOHomes     4.3  
147        Loiion     4.2  


# Saving Model

In [ ]:
import pickle

# Save the content-based filtering components
with open('path/content_based_model.pkl', 'wb') as f:
    pickle.dump({
        'tfidf_vectorizer': tfidf,
        'cosine_sim': cosine_sim,
        'content_scaler': content_scaler
    }, f)

# Save the collaborative filtering model
with open('path/collab_filter_model.pkl', 'wb') as f:
    pickle.dump(model, f)

Models and data saved successfully!


In [ ]:
import numpy as np
import pickle

# Load your dataset (assuming `df` is your dataframe)
num_users = 100  # Number of synthetic users

# Generate random user IDs and ratings
df['user_id'] = np.random.randint(1, num_users + 1, size=len(df))  # Random user IDs
df['user_rating'] = np.random.uniform(1, 5, size=len(df))  # Random ratings between 1 and 5

# Convert `user_id` to integer to avoid mismatches
df['user_id'] = df['user_id'].astype(int)

# Save the updated dataset
with open("path/dataset.pkl", "wb") as f:
    pickle.dump(df, f)

print("✅ Dataset saved successfully with user_id!")


✅ Dataset saved successfully with user_id!


In [ ]:
import pickle
import pandas as pd

# Load dataset.pkl
with open("path/dataset.pkl", "rb") as f:
    df = pickle.load(f)

# Print column names
print("🧐 Dataset columns:", df.columns.tolist())

# Check for user_id column
if "user_id" in df.columns:
    print(f"✅ 'user_id' column exists. Unique users: {df['user_id'].nunique()}")
else:
    print("❌ 'user_id' column is missing!")


🧐 Dataset columns: ['asin', 'title', 'brand', 'rating', 'image_url', 'description', 'user_id', 'user_rating']
✅ 'user_id' column exists. Unique users: 100
